In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# インストール

In [ ]:
!pip install -q keras-core --upgrade
!pip install -q keras-nlp --upgrade

# インポート

In [ ]:
import os
import pandas as pd
import keras_nlp
import keras_core as keras
from sklearn.model_selection import train_test_split

# バックエンドを TensorFlow に設定

In [ ]:
os.environ["KERAS_BACKEND"] = "tensorflow"

# データ読み込み

In [ ]:
df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df_test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

# トレイン　災害ではないのが０ 災害が１

In [ ]:
df_train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_train.head(50)

# トレイン　ツイート何行

In [ ]:
print(len(df_train))
df_train['target'].value_counts()

# テスト　ツイート何行

In [ ]:
df_test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
df_test.head()

print(len(df_test))

# 提出用ファイル みる

In [ ]:
df_sample = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
df_sample.head(50)

# データの情報、データの統計、何件欠損

In [ ]:
df_train.info()
df_train.describe()
df_train.isnull().sum()

# 欠損を埋める

In [ ]:
df_train["text"] = df_train["text"].fillna("")
df_test["text"] = df_test["text"].fillna("")

# データ分割

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(df_train["text"], df_train["target"], test_size=0.2, random_state=42)

#　学習

In [ ]:
classifier = keras_nlp.models.BertClassifier.from_preset(
    "bert_base_en_uncased",
    num_classes=2,
    activation="softmax"
)

import tensorflow as tf  

classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),  
    metrics=["accuracy"]
)

# 実験

In [ ]:
classifier.fit(
    X_train.to_list(), y_train,
    validation_data=(X_val.to_list(), y_val),
    epochs=1,
    batch_size=16
)

#  テスト用の予測 softmaxの最大値 → クラス番号

In [ ]:
test_probs = classifier.predict(df_test["text"].to_list(), batch_size=16)
test_preds = test_probs.argmax(axis=1)  # softmaxの最大値 → クラス番号

# 提出ファイルの作成

In [ ]:
submission = pd.DataFrame({
    "id": df_test["id"],
    "target": test_preds
})

submission.to_csv("submission2.csv", index=False)